# DocuSense
Semantic search, topic discovery and grounded question answering over BBC News articles.


In [1]:
# Upload the document collection.
from google.colab import files
uploaded = files.upload()


Saving BBC News Train.csv.zip to BBC News Train.csv.zip


In [2]:
%pip install -q sentence-transformers transformers


In [3]:
# Load and validate the article data.
from pathlib import Path
import pandas as pd


candidates = [name for name in uploaded if name.lower().endswith(('.csv', '.zip'))]
if len(candidates) != 1:
    raise ValueError("Upload exactly one BBC CSV or single-CSV ZIP file.")
df = pd.read_csv(candidates[0])
required = {'ArticleId', 'Text', 'Category'}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {required - set(df.columns)}")
df = df.dropna(subset=['Text']).copy()
df = df[df['Text'].astype(str).str.strip().ne('')].copy()
print(df.shape)
print(df['Category'].value_counts())


(1490, 3)
Category
sport            346
business         336
politics         274
entertainment    273
tech             261
Name: count, dtype: int64


In [4]:
print(df['Text'][0][:600])


worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness.  cynthia cooper  worldcom s ex-head of internal accounting  alerted directors to irregular accounting practices at the us telecoms giant in 2002. her warnings led to the collapse of the firm following the discovery of an $11bn (£5.7bn) accounting fraud. mr ebbers has pleaded not guilty to charges of fraud and conspiracy.  prosecution lawyers have argued that mr ebbers orchestrated a series of accounting tricks at 


In [5]:
# Inspect text lengths, missing values and duplicates.
df['word_count'] = df['Text'].str.split().str.len()
print(df['word_count'].describe())
print("Missing text:", df['Text'].isna().sum())
print("Duplicate texts:", df['Text'].duplicated().sum())


count    1490.000000
mean      385.012752
std       210.898616
min        90.000000
25%       253.000000
50%       337.000000
75%       468.750000
max      3345.000000
Name: word_count, dtype: float64
Missing text: 0
Duplicate texts: 50


In [6]:
# Remove duplicate articles and normalize whitespace.
import re


dups = df[df['Text'].duplicated(keep=False)]
print("Max distinct labels per duplicated text:", dups.groupby('Text')['Category'].nunique().max())


df = df.drop_duplicates(subset='Text').reset_index(drop=True)
print("Shape after dedupe:", df.shape)


df['clean_text'] = df['Text'].apply(lambda t: re.sub(r'\s+', ' ', t).strip())
print(df['clean_text'][0][:300])


Max distinct labels per duplicated text: 1
Shape after dedupe: (1440, 4)
worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness. cynthia cooper worldcom s ex-head of internal accounting alerted directors to irregular accounting practices at the u


In [7]:
# Create overlapping passages while retaining source identifiers.
def chunk_text(text, chunk_size=150, overlap=30):
    if not 0 <= overlap < chunk_size:
        raise ValueError("Require 0 <= overlap < chunk_size")
    words = text.split()
    # Overlap preserves context across passage boundaries.
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]
        chunks.append(" ".join(piece))
        if start + chunk_size >= len(words):
            break
    return chunks

rows = []
for _, r in df.iterrows():
    for i, c in enumerate(chunk_text(r['clean_text'])):
        rows.append({
            'chunk_id': f"{r['ArticleId']}_{i}",
            'ArticleId': r['ArticleId'],
            'Category': r['Category'],
            'chunk_text': c,
        })

chunks_df = pd.DataFrame(rows)
print(chunks_df.shape)
print(chunks_df['chunk_text'].str.split().str.len().describe())
print(chunks_df.iloc[0]['chunk_id'], '->', chunks_df.iloc[0]['chunk_text'][:200])


(4981, 4)
count    4981.000000
mean      132.692431
std        33.110327
min        31.000000
25%       136.000000
50%       150.000000
75%       150.000000
max       150.000000
Name: chunk_text, dtype: float64
1833_0 -> worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness. cynthia cooper wo


In [8]:
# Export the passage collection.
chunks_df.to_csv('bbc_chunks.csv', index=False)

from google.colab import files
files.download('bbc_chunks.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Build the TF-IDF passage index.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


vectorizer = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=2)
tfidf_matrix = vectorizer.fit_transform(chunks_df['chunk_text'])
print("Index shape (chunks x vocabulary):", tfidf_matrix.shape)

def search_tfidf(query, k=5):
    k = min(max(int(k), 0), len(chunks_df))
    if k == 0 or not query.strip():
        return chunks_df.iloc[:0].assign(score=0.0)
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).ravel()
    top = scores.argsort()[::-1][:k]
    results = chunks_df.iloc[top].copy()
    results['score'] = scores[top]
    return results[['chunk_id', 'ArticleId', 'Category', 'score', 'chunk_text']]

hits = search_tfidf("interest rates and inflation")
for _, h in hits.iterrows():
    print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
    print(h['chunk_text'][:200], "...\n")


Index shape (chunks x vocabulary): (4981, 16759)
[1088_4] (business) score=0.411
report published on wednesday the bank said that with rates at their current level inflation would rise to its 2% target within two years. however boe governor mervyn king warned only last month that  ...

[1874_0] (business) score=0.376
bank voted 8-1 for no rate change the decision to keep interest rates on hold at 4.75% earlier this month was passed 8-1 by the bank of england s rate-setting body minutes have shown. one member of th ...

[1874_1] (business) score=0.359
in interest rates to 5%. he argued that economic growth was picking up and that the equity credit and housing markets had been stronger than expected. the bank s minutes said that risks to the inflati ...

[1088_3] (business) score=0.318
work to do in 2005 said avery shenfeld senior economist at cibc world markets. i think the federal reserve does not want to rock the boat and is using a gradual approach in raising the interest rate s ...


In [10]:
# Compare keyword retrieval on example queries.
def show(query, k=3):
    print(f"\n=== {query} ===")
    for _, h in search_tfidf(query, k).iterrows():
        print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
        print(h['chunk_text'][:150], "...")

show("cost of borrowing money")
show("footballers signing new contracts")
show("computer viruses and online security")



=== cost of borrowing money ===
[1690_0] (politics) score=0.263
brown shrugs off economy fears gordon brown is to freeze petrol duty increases fund a £1bn package to avoid big council tax rises and boost childcare  ...
[192_1] (business) score=0.251
come. under direct threat will be the federal government s ability to make good on its debts to the social security trust fund. as soon as 2018 the fu ...
[334_0] (business) score=0.211
ecb holds rates amid growth fears the european central bank has left its key interest rate unchanged at 2% for the 19th month in succession. borrowing ...

=== footballers signing new contracts ===
[1127_2] (sport) score=0.153
senior appearances and scoring 25 goals before signing for sheffield united - where he scored 12 goals in 68 appearances. after signing for charlton h ...
[1831_4] (tech) score=0.151
and the first month under the new regime has seen many firms cautioned for not honouring them. some have been told to revamp websites so customers kno ..

In [11]:
!pip install -q sentence-transformers


In [12]:
# Encode passages with MiniLM and rank them by cosine similarity.
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(
    chunks_df['chunk_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(embeddings.shape)

def search_embed(query, k=5):
    k = min(max(int(k), 0), len(chunks_df))
    if k == 0 or not query.strip():
        return chunks_df.iloc[:0].assign(score=0.0)
    q = embedding_model.encode([query], normalize_embeddings=True)
    scores = (embeddings @ q.T).ravel()
    top = scores.argsort()[::-1][:k]
    results = chunks_df.iloc[top].copy()
    results['score'] = scores[top]
    return results[['chunk_id', 'ArticleId', 'Category', 'score', 'chunk_text']]

def show_embed(query, k=3):
    print(f"\n=== {query} ===")
    for _, h in search_embed(query, k).iterrows():
        print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
        print(h['chunk_text'][:150], "...")

show_embed("cost of borrowing money")
show_embed("footballers signing new contracts")
show_embed("computer viruses and online security")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

(4981, 384)

=== cost of borrowing money ===
[70_0] (business) score=0.481
indonesia declines debt freeze indonesia no longer needs the debt freeze offered by the paris club group of creditors economics minister aburizal bakr ...
[1690_4] (politics) score=0.479
and keep debt low and sustainable - both in this economic cycle and the next. borrowing figures for 2003/4 are £35bn - £2.5bn less than the £37.5bn pr ...
[248_2] (business) score=0.465
main topic of debate in the us presidential election. while demand for workers is far from booming the conditions are set for steady growth. overall c ...

=== footballers signing new contracts ===
[811_0] (sport) score=0.571
ronaldo considering new contract manchester united winger cristiano ronaldo said he is close to agreeing to a new contract at old trafford. the portug ...
[1588_0] (sport) score=0.523
wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008. wenger has ended speculation

In [13]:
# Save the passage embeddings.
import numpy as np
np.save('bbc_embeddings.npy', embeddings)

from google.colab import files
files.download('bbc_embeddings.npy')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
!pip install -q faiss-cpu


In [15]:
# Build a FAISS index over normalized embeddings.
import faiss

# Inner product equals cosine similarity for normalized vectors.
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype('float32'))
print("Vectors in index:", index.ntotal)

def search_faiss(query, k=5):
    k = min(max(int(k), 0), len(chunks_df))
    if k == 0 or not query.strip():
        return chunks_df.iloc[:0].assign(score=0.0)
    q = embedding_model.encode([query], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(q, k)
    results = chunks_df.iloc[ids[0]].copy()
    results['score'] = scores[0]
    return results[['chunk_id', 'ArticleId', 'Category', 'score', 'chunk_text']]

print(search_faiss("footballers signing new contracts", 3)[['chunk_id', 'Category', 'score']])


Vectors in index: 4981
     chunk_id Category     score
1213    811_0    sport  0.570594
50     1588_0    sport  0.523034
4802   1525_0    sport  0.463713


In [16]:
!pip install -q transformers accelerate


In [17]:
# Load FLAN-T5 separately from the retrieval encoder.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
lm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate(prompt, max_new_tokens=100):
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    out = lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [18]:
# Inspect selected source passages.
def read_chunk(cid, n=600):
    text = chunks_df.loc[chunks_df['chunk_id'] == cid, 'chunk_text'].iloc[0]
    print(f"--- {cid} ---\n{text[:n]}\n")

for cid in ['1057_1', '1057_2', '1588_0']:
    read_chunk(cid)

for cid in ['90_0', '90_1', '1874_0']:
    read_chunk(cid)


--- 1057_1 ---
at old trafford but the game was followed by a now notorious food fight which saw ferguson s clothes covered in soup and pizza. the sides meet again at highbury on 1 february. in the tunnel wenger was criticising my players calling them cheats so i told him to leave them alone and behave himself ferguson said on saturday. he ran at me with hands raised saying what do you want to do about it to not apologise for the behaviour of the players to another manager is unthinkable. it s a disgrace but i don t expect wenger to ever apologise he s that type of person. those allegations were put to wenge

--- 1057_2 ---
which arsenal lost to slip 10 points behind chelsea in the title race. at first he said only: i ve always been consistent with that story and told you nothing happened. if he has to talk he talks. if he wants to make a newspaper article he makes a newspaper article. he doesn t interest me and doesn t matter to me at all. i will never answer to any provocation from h

In [19]:
# Measure category agreement on example queries.
test_queries = {
    "business": ["Company fraud of millions of dollars", "Germeny's economic decline", "Positive or negative economies and the effect on families"],
    "sport": ["most unexpected sport come backs", "soccer players leaves his team", "recoveries from injuries from sports"],
    "tech": ["how to improve sales of gadgets", "motion control gaming controls", "internet speed in mb"],
    "politics": ["Corruption of uk keeping money from europe budget", "Iraq war when soldsmith wasnt advised", "noward truent snoker hall"],
    "entertainment": ["film director becomes officer", "meet the forckers film success in money", "channel audiences and what they watch more"],
}

def precision_at_k(search_fn, k=5):
    per_cat = {}
    for cat, queries in test_queries.items():
        scores = [(search_fn(q, k)['Category'] == cat).mean() for q in queries]
        per_cat[cat] = sum(scores) / len(scores)
    return per_cat, sum(per_cat.values()) / len(per_cat)

for name, fn in [("TF-IDF", search_tfidf), ("Embeddings (FAISS)", search_faiss)]:
    per_cat, overall = precision_at_k(fn)
    print(f"{name}: overall precision@5 = {overall:.3f}")
    for c, v in per_cat.items():
        print(f"   {c:14s} {v:.3f}")


TF-IDF: overall precision@5 = 0.627
   business       0.667
   sport          0.533
   tech           0.733
   politics       0.467
   entertainment  0.733
Embeddings (FAISS): overall precision@5 = 0.840
   business       0.800
   sport          1.000
   tech           0.933
   politics       0.667
   entertainment  0.800


In [20]:
# Inspect retrieval results for difficult queries.
for q in test_queries["politics"]:
    print(f"\n=== {q!r} ===")
    print("TF-IDF:")
    for _, h in search_tfidf(q, 5).iterrows():
        print(f"  [{h['chunk_id']}] ({h['Category']}) {h['chunk_text'][:80]}...")
    print("Embeddings:")
    for _, h in search_faiss(q, 5).iterrows():
        print(f"  [{h['chunk_id']}] ({h['Category']}) {h['chunk_text'][:80]}...")



=== 'Corruption of uk keeping money from europe budget' ===
TF-IDF:
  [1342_1] (business) was rampant during the shevardnadze era and boost georgia s fragile public finan...
  [1791_0] (politics) brown names 16 march for budget chancellor gordon brown will deliver his budget ...
  [1527_1] (politics) would be a political price to pay for the uk s stance - britain contributed more...
  [1534_2] (politics) to come to an end. mr straw said that as well as the veto over the rebate the uk...
  [192_0] (business) the ticking budget facing the us the budget proposals laid out by the administra...
Embeddings:
  [778_1] (politics) the court has now qualified its opinion on the community accounts for a decade i...
  [778_0] (politics) eu fraud clampdown urged eu member states are failing to report fraud and irregu...
  [778_2] (politics) eu presidency. any fraud in other member states is potentially fraud against the...
  [1534_0] (politics) uk rebate unjustified - chirac french president jacqu

In [21]:
!pip install -q pyspellchecker


In [22]:
# Test spelling correction with an English dictionary.
from spellchecker import SpellChecker

spell = SpellChecker()

def clean_query(query):
    words = query.lower().split()
    corrected = []
    for w in words:
        fix = spell.correction(w)
        corrected.append(fix if fix else w)
    return " ".join(corrected)


for q in ["noward truent snoker hall", "wenegr arsnal contrct", "soldsmith wasnt advised"]:
    print(f"{q!r} -> {clean_query(q)!r}")


'noward truent snoker hall' -> 'toward truest smoker hall'
'wenegr arsnal contrct' -> 'wiener arsenal contact'
'soldsmith wasnt advised' -> 'goldsmith want advised'


In [23]:
# Compare retrieval before and after spelling correction.
messy = "noward truent snoker hall"
fixed = clean_query(messy)
print("Fixed:", fixed)

print("\nTF-IDF before:", search_tfidf(messy, 5)['Category'].tolist())
print("TF-IDF after: ", search_tfidf(fixed, 5)['Category'].tolist())
print("\nEmbeddings before:", search_faiss(messy, 5)['Category'].tolist())
print("Embeddings after: ", search_faiss(fixed, 5)['Category'].tolist())


Fixed: toward truest smoker hall

TF-IDF before: ['politics', 'entertainment', 'tech', 'entertainment', 'entertainment']
TF-IDF after:  ['politics', 'entertainment', 'tech', 'entertainment', 'entertainment']

Embeddings before: ['sport', 'sport', 'sport', 'sport', 'sport']
Embeddings after:  ['entertainment', 'sport', 'entertainment', 'sport', 'sport']


In [24]:
# Extend spelling vocabulary with corpus terms.
from spellchecker import SpellChecker


corpus_words = " ".join(chunks_df['chunk_text'].tolist()).split()
spell = SpellChecker()
spell.word_frequency.load_words(corpus_words)

def clean_query_v2(query):
    words = query.lower().split()
    corrected = [spell.correction(w) or w for w in words]
    return " ".join(corrected)

fixed_v2 = clean_query_v2(messy)
print("Corrected (corpus-aware):", fixed_v2)

print("\nTF-IDF after v2:", search_tfidf(fixed_v2, 5)['chunk_id'].tolist())
print("Embeddings after v2:", search_faiss(fixed_v2, 5)['chunk_id'].tolist())


Corrected (corpus-aware): toward truest smoker hall

TF-IDF after v2: ['1582_0', '1167_0', '28_1', '2029_1', '714_2']
Embeddings after v2: ['714_6', '485_1', '714_5', '2173_4', '559_0']


In [25]:
# Check corpus-aware spelling corrections.
for messy in ["noward truent snoker hall", "wenegr arsnal contrct", "soldsmith wasnt advised"]:
    fixed = clean_query_v2(messy)
    print(f"{messy!r} -> {fixed!r}")


'noward truent snoker hall' -> 'toward truest smoker hall'
'wenegr arsnal contrct' -> 'wenger arsenal contact'
'soldsmith wasnt advised' -> 'goldsmith want advised'


In [26]:
# Normalize noun forms with WordNet.
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_query(query):
    return " ".join(lemmatizer.lemmatize(w) for w in query.lower().split())


for q in ["footballers signing new contracts", "players signed contracts", "companies raising prices"]:
    print(f"{q!r} -> {lemmatize_query(q)!r}")


'footballers signing new contracts' -> 'footballer signing new contract'
'players signed contracts' -> 'player signed contract'
'companies raising prices' -> 'company raising price'


In [27]:
# Build the spelling dictionary used by search.
from spellchecker import SpellChecker

corpus_words = " ".join(chunks_df['chunk_text'].tolist()).split()
spell_v2 = SpellChecker()
spell_v2.word_frequency.load_words(corpus_words)


In [28]:
# Query processing and grounded question answering.
def clean_query_full(query):
    """Spellcheck (corpus-aware) + lemmatize a raw user query."""
    corrected = " ".join(spell_v2.correction(w) or w for w in query.lower().split())
    return " ".join(lemmatizer.lemmatize(w) for w in corrected.split())

def search(query, method="embeddings", k=5, clean=True):
    """Search the corpus. method: 'tfidf' or 'embeddings'."""
    q = clean_query_full(query) if clean else query
    if method == "tfidf":
        return search_tfidf(q, k)
    elif method == "embeddings":
        return search_faiss(q, k)
    else:
        raise ValueError("method must be 'tfidf' or 'embeddings'")


import re
import torch

QA_MODEL_NAME = 'deepset/minilm-uncased-squad2'


# Provisional reader margin, not a probability.
QA_MIN_MARGIN = 2.0
NO_ANSWER = "I could not find a supported answer in the retrieved passages."
qa_tokenizer = None
qa_model = None
reranker = None

# Provisional passage-relevance cutoff.
RAG_MIN_RELEVANCE = 0.0


def load_qa_reader():
    """Load a reader trained to extract answers or predict no answer."""
    global qa_tokenizer, qa_model
    if qa_model is None:
        from transformers import AutoTokenizer, AutoModelForQuestionAnswering
        qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME, use_fast=True)
        qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)
        qa_model.eval()
    return qa_tokenizer, qa_model


def read_evidence(question, passage, max_answer_tokens=40):
    """Find a source-text span and compare its score with the no-answer score."""
    reader_tok, reader = load_qa_reader()
    if len(reader_tok(question, add_special_tokens=False)['input_ids']) > 96:
        raise ValueError('Please shorten the question to fewer than 96 tokens.')

    # Overlapping token windows retain context in long passages.
    encoded = reader_tok(
        question, passage, truncation='only_second', max_length=384,
        stride=96, return_overflowing_tokens=True,
        return_offsets_mapping=True, padding=True, return_tensors='pt'
    )
    offsets = encoded.pop('offset_mapping').tolist()
    encoded.pop('overflow_to_sample_mapping', None)
    device = next(reader.parameters()).device
    with torch.inference_mode():
        predictions = reader(**{key: value.to(device) for key, value in encoded.items()})
    best = None
    for window in range(len(offsets)):
        start_logits = predictions.start_logits[window].detach().cpu().tolist()
        end_logits = predictions.end_logits[window].detach().cpu().tolist()
        cls_pos = encoded['input_ids'][window].tolist().index(reader_tok.cls_token_id)
        null_score = start_logits[cls_pos] + end_logits[cls_pos]
        context_positions = [
            pos for pos, sequence in enumerate(encoded.sequence_ids(window))
            if sequence == 1 and offsets[window][pos][1] > offsets[window][pos][0]
        ]
        for start_index, start in enumerate(context_positions):
            for end in context_positions[start_index:start_index + max_answer_tokens]:
                margin = start_logits[start] + end_logits[end] - null_score
                if best is None or margin > best['margin']:
                    left, right = offsets[window][start][0], offsets[window][end][1]
                    best = {'text': passage[left:right], 'margin': float(margin),
                            'start': left, 'end': right}
    return best


def retrieve_rag(question, k=5, clean=False):
    """Retrieve candidates with both methods, then score question-passage relevance."""
    global reranker
    if not question.strip() or int(k) < 1:
        return []
    query = clean_query_full(question) if clean else question.strip()
    candidates = {}
    depth = min(max(int(k) * 4, 20), len(chunks_df))
    for search_fn in (search_faiss, search_tfidf):
        for _, row in search_fn(query, k=depth).iterrows():
            candidates.setdefault(str(row['chunk_id']), dict(row))
    passages = list(candidates.values())
    if not passages:
        return []
    if reranker is None:
        from sentence_transformers import CrossEncoder
        reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2', max_length=512)
    scores = reranker.predict(
        [(question, row['chunk_text']) for row in passages],
        batch_size=16, show_progress_bar=False, activation_fn=torch.nn.Identity()
    )
    for passage, score in zip(passages, scores):
        passage['relevance_score'] = float(score)
    return sorted(passages, key=lambda row: row['relevance_score'], reverse=True)[:int(k)]


def generate_grounded_phrase(question, evidence):
    """Use a simple QA prompt, with source identifiers added after generation."""

    prompt = f'question: {question}\ncontext: {evidence}\nshort answer:'
    encoded = tok(prompt, return_tensors='pt')
    if encoded['input_ids'].shape[-1] > 512:
        return evidence, 'extracted'
    device = next(lm.parameters()).device
    lm.eval()
    with torch.inference_mode():
        output = lm.generate(
            **{key: value.to(device) for key, value in encoded.items()},
            max_new_tokens=64, do_sample=False, num_beams=1,
            repetition_penalty=1.1, no_repeat_ngram_size=3
        )
    phrase = tok.decode(output[0], skip_special_tokens=True).strip()


    # Literal span containment checks provenance, not semantic correctness.
    normalize = lambda value: re.sub(r'\s+', ' ', value).strip().casefold()
    candidate = phrase.strip(' \"\'.,')
    if candidate and re.search(r'\w', candidate) and normalize(candidate) in normalize(evidence):
        return candidate, 'generated'
    return evidence, 'extracted'


def answer(question, k=5, clean=False, min_margin=QA_MIN_MARGIN):
    """Retrieve passages, check answerability and return a cited short answer."""
    question = question.strip()
    result = {'question': question, 'answer': NO_ANSWER, 'sources': [],
              'retrieved_sources': [], 'evidence': '', 'abstained': True,
              'mode': 'abstained', 'qa_margin': None, 'relevance_score': None}
    if not question:
        result['answer'] = 'Please enter a question.'
        return result
    hits = retrieve_rag(question, k=k, clean=clean)
    supported = []
    for hit in hits:
        label = f"ArticleId {hit['ArticleId']} ({hit['Category']}), chunk {hit['chunk_id']}"
        result['retrieved_sources'].append(label)

        if hit['relevance_score'] < RAG_MIN_RELEVANCE:
            continue
        evidence = read_evidence(question, hit['chunk_text'])
        if evidence and evidence['text'].strip() and evidence['margin'] >= min_margin:
            supported.append((evidence, hit, label))
    if not supported:
        return result

    evidence, hit, label = max(supported, key=lambda item: item[1]['relevance_score'])
    phrase, mode = generate_grounded_phrase(question, evidence['text'])
    result.update(
        answer=f'{phrase} [1]',
        sources=[f"[1] {label}\nSupporting passage: {hit['chunk_text']}\n"
                 f"Selected answer span: {evidence['text']}"],
        evidence=evidence['text'], abstained=False, mode=mode,
        qa_margin=evidence['margin'], relevance_score=hit['relevance_score']
    )
    return result


def answer_query_rag(query, top_k=5, max_length=150):
    """Console entry point; uses the same answer function as the interface."""
    result = answer(query, k=top_k)
    print(result['answer'])
    for source in result['sources']:
        print(source)
    if result['abstained']:
        print('Retrieved candidates (not supporting citations):')
        print('\n'.join(result['retrieved_sources']))
    return result


def answer_query_rag_v2(user_query, top_k=5, max_length=150):
    """Run the shared RAG console helper."""
    return answer_query_rag(user_query, top_k=top_k, max_length=max_length)


In [29]:
# Display an answer and its supporting source.
result = answer("What did Arsene Wenger agree to?")
print(result["answer"])
print("Answer mode:", result["mode"])
print("\n".join(result["sources"]))


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (7 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/minilm-uncased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


a long-term contract [1]
Answer mode: generated
[1] ArticleId 1588 (sport), chunk 1588_0
Supporting passage: wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008. wenger has ended speculation about his future by agreeing a long-term contract that takes him beyond the opening of arsenal s new stadium in two years. he said: signing a new contract just rubber-stamps my desire to take this club forward and fulfil my ambitions. i still have so much to achieve and my target is to drive this club on. these are exciting times for arsenal. the 55-year-old frenchman told arsenal s website www.arsenal.com: my intention has always been clear. i love this club and am very happy here. wenger has won the title and the fa cup three times each during his reign. chairman peter hill-wood said: we are absolutely delighted that arsene has signed an extension to his contract. since his arrival in 1996 he has
Selected answer span: a long-term contra

## Document retrieval evaluation
Precision and recall compare distinct articles using the same preprocessing and judgments. Initial scores use one expected article per query and remain provisional. Recall after pooled review is relative to the reviewed relevant articles.


In [30]:
# Evaluate distinct-document retrieval and collect relevance judgments.
labeled_queries = {
    "worldcom fraud trial bernie ebbers": 1833,
    "european central bank leaves rate unchanged": 334,
    "indonesia turns down debt freeze offer": 70,
    "wenegr arsnal contrct": 1588,
    "will ronaldo stay at man utd": 811,
    "gerrard wants to stay at liverpool": 1525,
    "soldsmith wasnt advised": 1683,
    "noward truent snoker hall": 1582,
    "is the uk eu rebate up for negotiation": 1527,
    "microsoft tools to clean viruses and spyware": 1439,
    "hi-tech thieves and malicious programs": 1186,
    "playstation games based on harry potter": 2205,
    "french director alan parker gets honoured by france for his work in film, what award did he receive and why was he chosen?": 1797,
    "did meet the fockers really break a christmas day box office record in north america or was it just doing better than other films at the time": 1866,
    "what happened with snoop dogg and the make up artist who sued him for rape, and how much money was she asking": 1407,
}


IR_K = 5
IR_CLEAN = True

def search_documents(query, method, k=IR_K):
    if int(k) < 1:
        raise ValueError('k must be at least 1')

    hits = search(query, method=method, k=len(chunks_df), clean=IR_CLEAN)
    return hits.drop_duplicates('ArticleId', keep='first').head(int(k)).copy()


def compare_retrievers(qrels, k=IR_K):
    """Compute macro precision/recall against the supplied positive article judgments."""
    rows = []
    for query, relevant_ids in qrels.items():
        relevant_ids = {str(article_id) for article_id in relevant_ids}
        if not relevant_ids:
            raise ValueError('Each evaluated query needs at least one relevant article.')
        for method in ('tfidf', 'embeddings'):
            hits = search_documents(query, method, k)
            found = {str(article_id) for article_id in hits['ArticleId']}
            matches = len(found & relevant_ids)
            rows.append(dict(query=query, method=method, k=k,
                relevant_found=matches, relevant_judged=len(relevant_ids),
                returned_articles=len(found), precision_at_k=matches/k,
                recall_at_k=matches/len(relevant_ids),
                returned_ids=' | '.join(str(value) for value in hits['ArticleId'])))
    detail = pd.DataFrame(rows)
    summary = detail.groupby('method')[['precision_at_k', 'recall_at_k']].mean()
    return detail, summary


seed_qrels = {query: {article_id} for query, article_id in labeled_queries.items()}
ir_detail, ir_summary = compare_retrievers(seed_qrels)
display(ir_summary)
print('PROVISIONAL: scores use one expected article per query, not exhaustive judgments.')
print('With one relevant ID, precision@5 cannot exceed 0.20; recall@5 is a hit rate.')
print('These development queries have already been inspected; use fresh queries for final testing.')
ir_detail.to_csv('ir_document_results.csv', index=False)
ir_summary.to_csv('ir_document_summary.csv')


judgment_rows = []
for query, seed_ids in seed_qrels.items():
    pooled = set(seed_ids)
    for method in ('tfidf', 'embeddings'):
        pooled.update(search_documents(query, method)['ArticleId'].tolist())
    for article_id in sorted(pooled, key=str):
        source = df.loc[df['ArticleId'] == article_id, 'clean_text']
        judgment_rows.append(dict(query=query, ArticleId=article_id,
            article_text=source.iloc[0] if len(source) else '',
            seed_relevant=int(article_id in seed_ids), relevant='', rationale=''))
ir_judgments = pd.DataFrame(judgment_rows)
ir_judgments.to_csv('ir_judgments_to_review.csv', index=False)
print('Review pooled articles and fill relevant with 1 or 0, plus a brief rationale.')
print('The seed_relevant field is a starting label; relevant stays blank until reviewed.')


def evaluate_reviewed_judgments(reviewed):
    """Recompute after every pooled candidate has a relevance judgment."""
    judgments = pd.to_numeric(reviewed['relevant'], errors='coerce')
    if not judgments.isin([0, 1]).all():
        raise ValueError('Every relevance judgment must be 0 or 1; review blank entries first.')
    reviewed = reviewed.assign(relevant=judgments)
    if reviewed.duplicated(['query', 'ArticleId']).any():
        raise ValueError('Keep one judgment per query and article.')
    expected_pairs = set(zip(ir_judgments['query'], ir_judgments['ArticleId'].astype(str)))
    supplied_pairs = set(zip(reviewed['query'], reviewed['ArticleId'].astype(str)))
    if not expected_pairs.issubset(supplied_pairs):
        raise ValueError('The reviewed file must include every pooled query/article pair.')
    qrels = {query: set(group.loc[group['relevant'] == 1, 'ArticleId'])
             for query, group in reviewed.groupby('query')}
    print('Recall is relative to reviewed positives; pooling can still miss relevant articles.')
    return compare_retrievers(qrels)


,precision_at_k,recall_at_k
method,,
embeddings,0.173333,0.866667
tfidf,0.200000,1.000000


PROVISIONAL: scores use one expected article per query, not exhaustive judgments.
With one relevant ID, precision@5 cannot exceed 0.20; recall@5 is a hit rate.
These development queries have already been inspected; use fresh queries for final testing.
Review pooled articles and fill relevant with 1 or 0, plus a brief rationale.
The seed_relevant field is a starting label; relevant stays blank until reviewed.


In [31]:
# Explain results using matched terms or similar sentences.
def matched_terms(query, chunk_text):
    q_vec = vectorizer.transform([query])
    feature_names = vectorizer.get_feature_names_out()
    q_terms = [feature_names[i] for i in q_vec.nonzero()[1]]
    text_words = set(chunk_text.lower().split())
    return [t for t in q_terms if t in text_words]

def most_similar_sentence(query, chunk_text):
    sentences = [s.strip() for s in chunk_text.split('. ') if s.strip()]
    if not sentences:
        return chunk_text[:100]
    sent_emb = embedding_model.encode(sentences, normalize_embeddings=True)
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    scores = (sent_emb @ q_emb.T).ravel()
    return sentences[scores.argmax()]

def search_explained(query, method="embeddings", k=5, clean=True):
    results = search(query, method=method, k=k, clean=clean)
    q = clean_query_full(query) if clean else query
    explanations = []
    for _, row in results.iterrows():
        if method == "tfidf":
            terms = matched_terms(q, row['chunk_text'])
            explanations.append(f"matched terms: {', '.join(terms)}" if terms else "no shared terms (weak match)")
        else:
            best = most_similar_sentence(q, row['chunk_text'])
            explanations.append(f"most similar sentence: \"{best[:120]}\"")
    results = results.copy()
    results['why'] = explanations
    return results[['chunk_id', 'Category', 'score', 'why']]

for method in ["tfidf", "embeddings"]:
    print(f"=== {method}, explained ===")
    for _, r in search_explained("wenegr arsnal contrct", method=method, k=3).iterrows():
        print(f"[{r['chunk_id']}] ({r['Category']}) score={r['score']:.3f}")
        print(f"  why: {r['why']}\n")


=== tfidf, explained ===
[1588_0] (sport) score=0.448
  why: matched terms: arsenal, wenger

[2061_0] (sport) score=0.430
  why: matched terms: arsenal, wenger

[1559_0] (sport) score=0.403
  why: matched terms: arsenal, wenger

=== embeddings, explained ===
[1057_2] (sport) score=0.644
  why: most similar sentence: "i have no diplomatic relations with him the arsenal boss is quoted as saying"

[1588_0] (sport) score=0.637
  why: most similar sentence: "wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008"

[1327_2] (sport) score=0.627
  why: most similar sentence: "with arsenal hit by injuries and suspension inexperienced midfield pair of mathieu flamini and cesc fabregas will line u"



In [32]:
# Compare retrieval methods and query corrections.
def demo(query, k=3):
    print("Raw query:    ", query)
    print("Cleaned query:", clean_query_full(query))
    for method in ["tfidf", "embeddings"]:
        print(f"\n--- {method} ---")
        for _, r in search_explained(query, method=method, k=k).iterrows():
            print(f"[{r['chunk_id']}] ({r['Category']}) score={r['score']:.3f}")
            print(f"  why: {r['why']}")

demo("wenegr arsnal contrct")


Raw query:     wenegr arsnal contrct
Cleaned query: wenger arsenal contact

--- tfidf ---
[1588_0] (sport) score=0.448
  why: matched terms: arsenal, wenger
[2061_0] (sport) score=0.430
  why: matched terms: arsenal, wenger
[1559_0] (sport) score=0.403
  why: matched terms: arsenal, wenger

--- embeddings ---
[1057_2] (sport) score=0.644
  why: most similar sentence: "i have no diplomatic relations with him the arsenal boss is quoted as saying"
[1588_0] (sport) score=0.637
  why: most similar sentence: "wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008"
[1327_2] (sport) score=0.627
  why: most similar sentence: "with arsenal hit by injuries and suspension inexperienced midfield pair of mathieu flamini and cesc fabregas will line u"


In [33]:
# Discover document topics with BERTopic.
%pip install -q bertopic
from bertopic import BERTopic

TEXT_COLUMN = 'Text'


topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",
    min_topic_size=15,
    verbose=True
)


topics, probs = topic_model.fit_transform(df[TEXT_COLUMN].tolist())


df['bertopic_id'] = topics


topic_info = topic_model.get_topic_info()
print(f"\n✅ Successfully discovered {len(topic_info) - 1} sub-topics (excluding outliers)!\n")
print(topic_info.head(10)[['Topic', 'Count', 'Name']])


2026-11-03 20:59:47,106 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

2026-11-03 20:59:53,852 - BERTopic - Embedding - Completed ✓
2026-11-03 20:59:53,855 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-03 21:00:10,908 - BERTopic - Dimensionality - Completed ✓
2026-11-03 21:00:10,909 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-03 21:00:10,953 - BERTopic - Cluster - Completed ✓
2026-11-03 21:00:10,957 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-03 21:00:11,395 - BERTopic - Representation - Completed ✓



✅ Successfully discovered 9 sub-topics (excluding outliers)!

   Topic  Count                       Name
0     -1     20           -1_the_of_to_and
1      0    298             0_the_in_to_of
2      1    273            1_the_and_of_in
3      2    243            2_the_to_of_and
4      3    230            3_the_to_of_and
5      4    230            4_the_to_and_in
6      5     59            5_the_in_to_and
7      6     44            6_the_in_she_to
8      7     22  7_the_kenteris_iaaf_conte
9      8     21     8_yukos_the_russian_of


In [34]:
# Remove English stop words from topic keywords.
from sklearn.feature_extraction.text import CountVectorizer


vectorizer_model = CountVectorizer(stop_words="english")
topic_model.update_topics(df['Text'].tolist(), vectorizer_model=vectorizer_model)


topic_info = topic_model.get_topic_info()
print("=== CLEANED BERTOPIC SUB-TOPICS ===")
print(topic_info.head(12)[['Topic', 'Count', 'Name']])


=== CLEANED BERTOPIC SUB-TOPICS ===
   Topic  Count                             Name
0     -1     20  -1_eu_patent_directive_european
1      0    298        0_said_year_growth_market
2      1    273           1_film_best_said_music
3      2    243        2_mr_said_labour_election
4      3    230       3_said_people_mobile_users
5      4    230        4_england_game_club_wales
6      5     59   5_roddick_open_seed_australian
7      6     44      6_olympic_world_holmes_race
8      7     22     7_kenteris_iaaf_conte_doping
9      8     21   8_yukos_russian_russia_gazprom


In [35]:
# Remove news filler words from topic keywords.
from sklearn.feature_extraction.text import CountVectorizer


custom_stop_words = list(CountVectorizer(stop_words="english").get_stop_words()) + [
    "said", "say", "says", "mr", "year", "years", "people", "new", "time", "would", "also", "one"
]


vectorizer_model = CountVectorizer(stop_words=custom_stop_words)
topic_model.update_topics(df['Text'].tolist(), vectorizer_model=vectorizer_model)


topic_info = topic_model.get_topic_info()
print("=== BERTOPIC TOPIC KEYWORDS ===")
print(topic_info.head(12)[['Topic', 'Count', 'Name']])


=== BERTOPIC TOPIC KEYWORDS ===
   Topic  Count                                Name
0     -1     20     -1_eu_patent_directive_european
1      0    298    0_growth_market_economy_economic
2      1    273            1_film_best_music_awards
3      2    243  2_labour_election_government_blair
4      3    230     3_mobile_users_technology_phone
5      4    230           4_england_game_club_wales
6      5     59      5_roddick_open_seed_australian
7      6     44         6_olympic_world_holmes_race
8      7     22        7_kenteris_iaaf_conte_doping
9      8     21      8_yukos_russian_russia_gazprom


In [36]:
# Reuse the loaded answer-generation model.
tokenizer = tok
generation_model = lm


In [37]:
# Run the shared RAG console helper.
result = answer_query_rag_v2("What did Arsene Wenger agree to?")


a long-term contract [1]
[1] ArticleId 1588 (sport), chunk 1588_0
Supporting passage: wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008. wenger has ended speculation about his future by agreeing a long-term contract that takes him beyond the opening of arsenal s new stadium in two years. he said: signing a new contract just rubber-stamps my desire to take this club forward and fulfil my ambitions. i still have so much to achieve and my target is to drive this club on. these are exciting times for arsenal. the 55-year-old frenchman told arsenal s website www.arsenal.com: my intention has always been clear. i love this club and am very happy here. wenger has won the title and the fa cup three times each during his reign. chairman peter hill-wood said: we are absolutely delighted that arsene has signed an extension to his contract. since his arrival in 1996 he has
Selected answer span: a long-term contract that takes him beyon

## RAG grounding checks
Development checks cover answerable and unanswerable examples. Passage relevance and reader margins are provisional scores, not probabilities. Matching text or finding an answer span does not establish semantic correctness.
Models: [MiniLM SQuAD2 reader](https://huggingface.co/deepset/minilm-uncased-squad2) and [MS MARCO reranker](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2).


In [38]:
# Check answer extraction on fixed passages.
fixture_passage = (
    "Arsenal manager Arsene Wenger signed a new contract to stay at the club "
    "until May 2008. He agreed to a long-term contract."
)
reader_checks = []
for question, expected, should_abstain in [
    ("Until when did Wenger sign a contract to stay at Arsenal?", "May 2008", False),
    ("What is Wenger's private bank account number?", None, True),
]:
    prediction = read_evidence(question, fixture_passage)
    abstained = prediction is None or prediction["margin"] < QA_MIN_MARGIN
    passed = abstained if should_abstain else (
        not abstained and expected.casefold() in prediction["text"].casefold()
    )
    reader_checks.append({"question": question, "prediction": prediction,
                          "abstained": abstained, "passed": passed})
display(pd.DataFrame(reader_checks))


,question,prediction,abstained,passed
0,Until when did Wenger sign a contract to stay ...,"{'text': 'May 2008', 'margin': 15.363068997859...",False,True
1,What is Wenger's private bank account number?,{'text': 'Arsenal manager Arsene Wenger signed...,True,True


In [39]:
# Record RAG development checks across topics and unanswerable questions.
rag_cases = [
    ("sport", "What did Arsene Wenger agree to?", 1588, False),
    ("sport", "Until when did Arsene Wenger sign a contract to stay at Arsenal?", 1588, False),
    ("business", "At what percentage did the European Central Bank leave its key interest rate?", 334, False),
    ("business", "Which country turned down a debt freeze offered by the Paris Club?", 70, False),
    ("tech", "What is Microsoft releasing to clean PCs of viruses and spyware?", 1439, False),
    ("politics", "Which country's EU rebate was being discussed for negotiation?", 1527, False),
    ("entertainment", "Which film director was honoured by France?", 1797, False),
    ("unanswerable", "What is Arsene Wenger's private bank account number?", None, True),
    ("unanswerable", "What is the secret administrator password for Microsoft's internal payroll system?", None, True),
    ("unanswerable", "Who won the 2099 FIFA World Cup final?", None, True),
]
rag_rows = []
for category, question, reference_id, expected_abstention in rag_cases:
    response = answer(question)
    cited_ids = [int(value) for value in re.findall(
        r"ArticleId (\d+)", "\n".join(response["sources"]))]
    print(f"\n[{category}] {question}\n{response['answer']}")
    print("Mode:", response["mode"])
    print("\n".join(response["sources"]))
    rag_rows.append({
        "category": category, "question": question,
        "reference_article_candidate": reference_id,
        "expected_abstention": expected_abstention,
        "actual_answer": response["answer"], "mode": response["mode"],
        "abstention_check": response["abstained"] == expected_abstention,
        "cited_article_ids": str(cited_ids),
        "reference_cited": reference_id in cited_ids if reference_id is not None else None,
        "selected_span": response["evidence"], "sources": "\n".join(response["sources"]),
        "retrieved_candidates": "\n".join(response["retrieved_sources"]),
        "qa_margin": response["qa_margin"],
        "relevance_score": response["relevance_score"],
        "human_answer_correct": "", "human_citation_supports_answer": "", "human_notes": "",
    })
rag_checks = pd.DataFrame(rag_rows)
rag_checks.to_csv("rag_grounding_checks.csv", index=False)
display(rag_checks[["category", "question", "actual_answer", "mode", "abstention_check"]])
print("Abstention checks:", int(rag_checks["abstention_check"].sum()), "/", len(rag_checks))
print("Abstention checks only measure whether an answer was attempted, not whether it is correct.")


reference_articles = df[df["ArticleId"].isin(
    [case[2] for case in rag_cases if case[2] is not None]
)][["ArticleId", "Category", "clean_text"]]
display(reference_articles)



[sport] What did Arsene Wenger agree to?
a long-term contract [1]
Mode: generated
[1] ArticleId 1588 (sport), chunk 1588_0
Supporting passage: wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008. wenger has ended speculation about his future by agreeing a long-term contract that takes him beyond the opening of arsenal s new stadium in two years. he said: signing a new contract just rubber-stamps my desire to take this club forward and fulfil my ambitions. i still have so much to achieve and my target is to drive this club on. these are exciting times for arsenal. the 55-year-old frenchman told arsenal s website www.arsenal.com: my intention has always been clear. i love this club and am very happy here. wenger has won the title and the fa cup three times each during his reign. chairman peter hill-wood said: we are absolutely delighted that arsene has signed an extension to his contract. since his arrival in 1996 he has
Selec

,category,question,actual_answer,mode,abstention_check
0,sport,What did Arsene Wenger agree to?,a long-term contract [1],generated,True
1,sport,Until when did Arsene Wenger sign a contract t...,may 2008 [1],extracted,True
2,business,At what percentage did the European Central Ba...,2% [1],generated,True
3,business,Which country turned down a debt freeze offere...,indonesia [1],generated,True
4,tech,What is Microsoft releasing to clean PCs of vi...,security tools [1],extracted,True
5,politics,Which country's EU rebate was being discussed ...,luxembourg [1],extracted,True
6,entertainment,Which film director was honoured by France?,sir alan parker [1],extracted,True
7,unanswerable,What is Arsene Wenger's private bank account n...,I could not find a supported answer in the ret...,abstained,True
8,unanswerable,What is the secret administrator password for ...,I could not find a supported answer in the ret...,abstained,True
9,unanswerable,Who won the 2099 FIFA World Cup final?,I could not find a supported answer in the ret...,abstained,True


Abstention checks: 10 / 10
Abstention checks only measure whether an answer was attempted, not whether it is correct.


,ArticleId,Category,clean_text
7,1797,entertainment,french honour for director parker british film...
15,1588,sport,wenger signs new deal arsenal manager arsene w...
28,1527,politics,uk will stand firm on eu rebate britain s £3bn...
109,334,business,ecb holds rates amid growth fears the european...
274,1439,tech,microsoft debuts security tools microsoft is r...
1066,70,business,indonesia declines debt freeze indonesia no lo...


## Answer correctness and source support
Phrase matching is an automatic development check. Correctness and citation support require separate passage-level judgments. The debt-freeze question admits multiple countries, and the saved rebate response confuses the UK with Luxembourg.


In [40]:
# Compare expected phrases and summarize reviewed answers.
expected_answers = {
    "What did Arsene Wenger agree to?": ["long-term contract", "new contract"],
    "Until when did Arsene Wenger sign a contract to stay at Arsenal?": ["May 2008"],
    "At what percentage did the European Central Bank leave its key interest rate?": ["2%", "2 percent", "two percent"],
    "Which country turned down a debt freeze offered by the Paris Club?": ["Indonesia", "Thailand", "India"],
    "What is Microsoft releasing to clean PCs of viruses and spyware?": ["security tools", "tools", "anti-virus", "antivirus", "anti-spyware"],
    "Which country's EU rebate was being discussed for negotiation?": ["UK", "United Kingdom", "Britain"],
    "Which film director was honoured by France?": ["Alan Parker"],
}

def normalize_answer(text):
    text = re.sub(r"\[\d+\]", "", str(text)).casefold()
    return re.sub(r"\s+", " ", re.sub(r"[^\w%]+", " ", text)).strip()

def matches_expected_phrase(text, aliases):
    normalized = normalize_answer(text)
    return any(re.search(r"(?<!\w)" + re.escape(normalize_answer(alias)) + r"(?!\w)", normalized)
               for alias in aliases)

rag_review = rag_checks.copy()
rag_review["expected_answer_guidance"] = rag_review["question"].map(
    lambda question: " / ".join(expected_answers.get(question, [])) or "No supported answer expected")
rag_review["phrase_match_only"] = rag_review.apply(
    lambda row: None if row["expected_abstention"] else matches_expected_phrase(
        row["actual_answer"], expected_answers[row["question"]]), axis=1)

rag_review.to_csv("rag_grounding_review.csv", index=False)
display(rag_review[["question", "expected_answer_guidance", "actual_answer",
                    "phrase_match_only", "abstention_check"]])
answerable = rag_review[~rag_review["expected_abstention"]]
unanswerable = rag_review[rag_review["expected_abstention"]]
print(f"Expected-phrase matches: {int(answerable['phrase_match_only'].sum())}/{len(answerable)} answerable examples")
print(f"Appropriate abstentions: {int(unanswerable['abstention_check'].sum())}/{len(unanswerable)} unanswerable examples")
print("These are development checks. Phrase matching is not a grounded-answer accuracy metric.")
print("Review each cited passage, then complete the human-review columns in rag_grounding_review.csv.")


def summarize_manual_review(review):
    def flag(value):
        return {"true": True, "yes": True, "1": True,
                "false": False, "no": False, "0": False}.get(str(value).strip().casefold())
    for column, label in [
        ("human_answer_correct", "Answer correctness on answerable questions"),
        ("human_citation_supports_answer", "Citation support on answerable questions"),
    ]:
        values = [flag(value) for value in review.loc[~review["expected_abstention"], column]]
        judged = [value for value in values if value is not None]
        print(f"{label}: {sum(judged)}/{len(judged)} reviewed; {len(values)-len(judged)} pending")

summarize_manual_review(rag_review)


,question,expected_answer_guidance,actual_answer,phrase_match_only,abstention_check
0,What did Arsene Wenger agree to?,long-term contract / new contract,a long-term contract [1],True,True
1,Until when did Arsene Wenger sign a contract t...,May 2008,may 2008 [1],True,True
2,At what percentage did the European Central Ba...,2% / 2 percent / two percent,2% [1],True,True
3,Which country turned down a debt freeze offere...,Indonesia / Thailand / India,indonesia [1],True,True
4,What is Microsoft releasing to clean PCs of vi...,security tools / tools / anti-virus / antiviru...,security tools [1],True,True
5,Which country's EU rebate was being discussed ...,UK / United Kingdom / Britain,luxembourg [1],False,True
6,Which film director was honoured by France?,Alan Parker,sir alan parker [1],True,True
7,What is Arsene Wenger's private bank account n...,No supported answer expected,I could not find a supported answer in the ret...,None,True
8,What is the secret administrator password for ...,No supported answer expected,I could not find a supported answer in the ret...,None,True
9,Who won the 2099 FIFA World Cup final?,No supported answer expected,I could not find a supported answer in the ret...,None,True


Expected-phrase matches: 6/7 answerable examples
Appropriate abstentions: 3/3 unanswerable examples
These are development checks. Phrase matching is not a grounded-answer accuracy metric.
Review each cited passage, then complete the human-review columns in rag_grounding_review.csv.
Answer correctness on answerable questions: 0/0 reviewed; 7 pending
Citation support on answerable questions: 0/0 reviewed; 7 pending


In [ ]:
# Browse example articles from a discovered topic.
def browse_bertopic(topic_id, top_n=3):
    """
    Displays sample document passages belonging to a specific BERTopic cluster.
    Note: Topic -1 in BERTopic represents unclustered/outlier documents.
    """
    if topic_id not in df['bertopic_id'].values:
        print(f"Topic {topic_id} not found.")
        return

    keywords = [word for word, _ in topic_model.get_topic(topic_id)]
    filtered = df[df['bertopic_id'] == topic_id]

    print(f"=== BROWSING TOPIC {topic_id} ===")
    print(f"Keywords: {', '.join(keywords[:5])}")
    print(f"Total Articles in Topic: {len(filtered)}\n")

    for idx, row in filtered.head(top_n).iterrows():
        print(f"• Article {row['ArticleId']} ({row['Category']}):")
        print(f"  \"{str(row['Text'])[:200]}...\"\n")


browse_bertopic(topic_id=0, top_n=3)


## Evaluation scope
Results describe inspected development examples. Independent test questions and completed relevance judgments are still needed for final performance claims.


In [ ]:
# Build the search and question-answering interface.
!pip install -q gradio
import gradio as gr

# Optional evaluation controls.
SHOW_REVIEW_TOOLS = True

text_lookup = dict(zip(chunks_df['chunk_id'], chunks_df['chunk_text']))

css = """
.gradio-container,
.gradio-container.dark,
.dark .gradio-container {
    color-scheme: light;
    color: #111827 !important;
    background: linear-gradient(135deg, #eef2ff 0%, #faf5ff 50%, #fff1f2 100%) !important;
    font-family: Inter, system-ui, sans-serif !important;
    --body-background-fill: #f8fafc !important;
    --body-text-color: #111827 !important;
    --body-text-color-subdued: #374151 !important;
    --background-fill-primary: #ffffff !important;
    --background-fill-secondary: #f1f5f9 !important;
    --block-background-fill: #ffffff !important;
    --block-label-background-fill: #ffffff !important;
    --block-label-text-color: #1f2937 !important;
    --block-title-text-color: #111827 !important;
    --input-background-fill: #ffffff !important;
    --input-text-color: #111827 !important;
    --input-placeholder-color: #4b5563 !important;
    --table-text-color: #111827 !important;
    --table-even-background-fill: #ffffff !important;
    --table-odd-background-fill: #f8fafc !important;
    --button-secondary-background-fill: #eef2ff !important;
    --button-secondary-background-fill-hover: #e0e7ff !important;
    --button-secondary-text-color: #1f2937 !important;
    --button-primary-text-color: #ffffff !important;
    --checkbox-label-background-fill: #ffffff !important;
    --checkbox-label-background-fill-hover: #eef2ff !important;
    --checkbox-label-background-fill-selected: #e0e7ff !important;
    --checkbox-label-text-color: #111827 !important;
    --checkbox-label-text-color-selected: #111827 !important;
    --tw-prose-body: #1f2937 !important;
}
.gradio-container :is(label, legend, p, h2, h3, h4, .label-text, .prose, .caption),
.gradio-container :is(label, legend) :is(span, div) {
    color: #1f2937 !important;
    opacity: 1 !important;
}
.gradio-container :is(textarea, input:not([type=range]):not([type=radio]):not([type=checkbox])) {
    color: #111827 !important;
    -webkit-text-fill-color: #111827 !important;
    background: #ffffff !important;
    caret-color: #111827;
    opacity: 1 !important;
}
.gradio-container :is(input, textarea)::placeholder {
    color: #4b5563 !important;
    -webkit-text-fill-color: #4b5563 !important;
    opacity: 1 !important;
}
.gradio-container :is(th, td, [role=gridcell], [role=columnheader]),
.gradio-container :is(th, td, [role=gridcell], [role=columnheader]) :is(span, div),
.gradio-container button, .gradio-container button span {
    color: #1f2937 !important;
    -webkit-text-fill-color: #1f2937 !important;
    opacity: 1 !important;
}
.gradio-container :is(.tab-nav, [role=tablist]) {
    background: #ffffff !important;
    border-bottom: 1px solid #cbd5e1 !important;
}
.gradio-container .tab-nav button, .gradio-container button[role=tab] {
    background: #f1f5f9 !important;
    border: 1px solid #cbd5e1 !important;
    font-weight: 600 !important;
}
.gradio-container .tab-nav button.selected,
.gradio-container .tab-nav button.selected span,
.gradio-container button[role=tab][aria-selected=true],
.gradio-container button[role=tab][aria-selected=true] span {
    color: #ffffff !important;
    -webkit-text-fill-color: #ffffff !important;
    background: #4338ca !important;
    border-color: #4338ca !important;
}
.gradio-container .go-btn, .gradio-container .go-btn span {
    background: #4338ca !important;
    color: #ffffff !important;
    -webkit-text-fill-color: #ffffff !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: 600 !important;
}
.gradio-container button:focus-visible {
    outline: 3px solid #111827 !important;
    outline-offset: 2px !important;
}
.gradio-container .card {
    background: #ffffff !important;
    border-radius: 16px !important;
    border: 1px solid #cbd5e1 !important;
    padding: 18px !important;
}
.gradio-container .hero { text-align: center; padding: 28px 10px 6px; }
.gradio-container .hero h1 {
    color: #312e81 !important;
    -webkit-text-fill-color: #312e81 !important;
    font-size: 2.6rem; margin: 0; font-weight: 800;
}
.gradio-container .hero p { color: #374151 !important; }
footer { display: none !important; }
"""
theme = gr.themes.Soft(primary_hue="indigo", secondary_hue="purple", neutral_hue="slate")

def ui_search(query, method, k):
    if not query.strip():
        return "", []
    res = search_explained(query, method=method, k=int(k))
    rows = [[r['chunk_id'], r['Category'], round(float(r['score']), 3),
             text_lookup[r['chunk_id']][:180] + "...", r['why']]
            for _, r in res.iterrows()]
    return clean_query_full(query), rows

def ui_answer(question):
    if not question.strip():
        return "", ""
    r = answer(question)
    sources = [source.split("\nSelected answer span:", 1)[0] for source in r["sources"]]
    return r["answer"], "\n\n".join(sources)


from pathlib import Path
REVIEW_EXPORTS = (Path.cwd() / 'docusense_exports').resolve()
REVIEW_FILE = str(REVIEW_EXPORTS / 'ir_judgments_reviewed.csv')
if SHOW_REVIEW_TOOLS:
    REVIEW_EXPORTS.mkdir(parents=True, exist_ok=True)
    if not Path(REVIEW_FILE).exists():
        ir_judgments.to_csv(REVIEW_FILE, index=False)


import json
CSV_DOWNLOAD_JS = '(csv) => {\n    if (typeof csv !== "string" || csv.trim() === "") {\n        return "No saved CSV is available to download.";\n    }\n    try {\n        const blob = new Blob(["\\ufeff", csv], {type: "text/csv;charset=utf-8"});\n        const url = URL.createObjectURL(blob);\n        const anchor = document.createElement("a");\n        anchor.href = url;\n        anchor.download = __FILENAME__;\n        document.body.appendChild(anchor);\n        anchor.click();\n        anchor.remove();\n        setTimeout(() => URL.revokeObjectURL(url), 30000);\n        return "Download requested: " + __FILENAME__ + ". Check your browser downloads.";\n    } catch (error) {\n        return "Download could not start: " + error.message;\n    }\n}'

def csv_download_js(filename):
    return CSV_DOWNLOAD_JS.replace('__FILENAME__', json.dumps(filename))


def review_load(record):
    row = ir_judgments.iloc[int(record)]
    value = str(row['relevant']).strip()
    label = {'0': 'Not relevant', '1': 'Relevant', '0.0': 'Not relevant', '1.0': 'Relevant'}.get(value, 'Unreviewed')
    note = '' if pd.isna(row['rationale']) else str(row['rationale'])
    return row['query'], f"ArticleId {row['ArticleId']}", row['article_text'], label, note


def review_save(record, judgment, rationale):
    if judgment not in ('Relevant', 'Not relevant'):
        return 'Not saved: choose Relevant or Not relevant.', ir_judgments.to_csv(index=False)
    rationale = str(rationale or '').strip()
    if not rationale:
        return 'Not saved: add a brief reason for your judgment.', ir_judgments.to_csv(index=False)
    index = ir_judgments.index[int(record)]

    ir_judgments['relevant'] = ir_judgments['relevant'].astype('string')
    ir_judgments['rationale'] = ir_judgments['rationale'].astype('string')
    ir_judgments.at[index, 'relevant'] = '1' if judgment == 'Relevant' else '0'
    ir_judgments.at[index, 'rationale'] = rationale
    ir_judgments.to_csv(REVIEW_FILE, index=False)
    complete = pd.to_numeric(ir_judgments['relevant'], errors='coerce').isin([0, 1]).sum()
    return f'Saved. {complete}/{len(ir_judgments)} articles reviewed. Click Download current judgments to keep a copy.', ir_judgments.to_csv(index=False)


def review_scores():
    try:
        detail, summary = evaluate_reviewed_judgments(ir_judgments)
    except ValueError as error:
        raise gr.Error(str(error))
    details_path = str(REVIEW_EXPORTS / 'ir_reviewed_results.csv')
    summary_path = str(REVIEW_EXPORTS / 'ir_reviewed_summary.csv')
    detail.to_csv(details_path, index=False)
    summary.to_csv(summary_path)
    return (summary.reset_index(), detail.to_csv(index=False), summary.to_csv(),
            gr.update(interactive=True), gr.update(interactive=True))


if 'app' in globals():
    app.close()

try:
    app = gr.Blocks(title="DocuSense", theme=theme, css=css)
except TypeError:
    app = gr.Blocks(title="DocuSense")

with app:
    gr.HTML(f"""<style>{css}</style>
    <div class='hero'>
      <h1>DocuSense</h1>
      <p>Search {len(df):,} articles</p>
    </div>""")

    with gr.Tab("Search"):
        with gr.Column(elem_classes="card"):
            with gr.Row():
                q = gr.Textbox(label="What are you looking for?", scale=4,
                               placeholder="e.g. wenegr arsnal contrct")
                btn = gr.Button("Search", elem_classes="go-btn", scale=1)
            with gr.Row():
                method = gr.Radio([("Semantic search", "embeddings"), ("Keyword search", "tfidf")], value="embeddings", label="Search method")
                k = gr.Slider(1, 10, value=5, step=1, label="Number of results")
            gr.Examples(
                examples=[["wenegr arsnal contrct"], ["who is suing snoop dogg"],
                          ["microsoft tools to remove spyware"], ["bank of england interest rates"]],
                inputs=[q], label="Try one of these")
        with gr.Column(elem_classes="card"):
            cleaned = gr.Textbox(label="Processed query", interactive=False)
            table = gr.Dataframe(
                headers=["Source", "Category", "Similarity", "Passage", "Why this result"],
                wrap=True, interactive=False)
        btn.click(ui_search, [q, method, k], [cleaned, table])
        q.submit(ui_search, [q, method, k], [cleaned, table])

    with gr.Tab("Ask a question"):
        with gr.Column(elem_classes="card"):
            with gr.Row():
                question = gr.Textbox(label="Ask about the articles", scale=4,
                                      placeholder="e.g. What did Arsene Wenger agree to?")
                ask = gr.Button("Answer", elem_classes="go-btn", scale=1)
            gr.Examples(
                examples=[["What did Arsene Wenger agree to?"],
                          ["What is Microsoft releasing to deal with viruses?"]],
                inputs=[question], label="Try one of these")
        with gr.Column(elem_classes="card"):
            ans = gr.Textbox(label="Answer", interactive=False)
            src = gr.Textbox(label="Sources and supporting passages", interactive=False)
        ask.click(ui_answer, [question], [ans, src])
        question.submit(ui_answer, [question], [ans, src])

    if SHOW_REVIEW_TOOLS:
        with gr.Tab("Review retrieval"):
            gr.Markdown("Read each query and full article. Mark relevance to the query, not just its category. Seed labels are hidden to avoid anchoring the review. Save and download your judgments before closing Colab.")
            review_record = gr.Slider(0, len(ir_judgments)-1, value=0, step=1, label="Candidate record (move to another record after saving)")
            initial = review_load(0)
            review_question = gr.Textbox(value=initial[0], label="Query", interactive=False)
            review_id = gr.Textbox(value=initial[1], label="Source", interactive=False)
            review_article = gr.Textbox(value=initial[2], label="Full article", lines=14, interactive=False)
            review_judgment = gr.Radio(['Unreviewed', 'Relevant', 'Not relevant'], value=initial[3], label="Does the article address the query?")
            review_reason = gr.Textbox(value=initial[4], label="Reason for your judgment")
            review_save_button = gr.Button("Save judgment")
            review_status = gr.Textbox(label="Review progress", interactive=False)
            review_csv_payload = gr.Textbox(value=ir_judgments.to_csv(index=False), visible=False)
            review_file = gr.Button("Download current judgments")
            review_download_status = gr.Textbox(label="Download status", interactive=False)
            review_file.click(fn=None, inputs=[review_csv_payload], outputs=[review_download_status],
                              js=csv_download_js("ir_judgments_reviewed.csv"), queue=False)
            review_record.change(review_load, [review_record], [review_question, review_id, review_article, review_judgment, review_reason])
            review_save_button.click(review_save, [review_record, review_judgment, review_reason], [review_status, review_csv_payload])
            review_score_button = gr.Button("Calculate scores after all candidates are reviewed")
            review_table = gr.Dataframe(label="Precision and recall from reviewed judgments", interactive=False)
            review_results_payload = gr.Textbox(visible=False)
            review_results_file = gr.Button("Download reviewed query results", interactive=False)
            review_results_file.click(fn=None, inputs=[review_results_payload], outputs=[review_download_status],
                                      js=csv_download_js("ir_reviewed_results.csv"), queue=False)
            review_summary_payload = gr.Textbox(visible=False)
            review_summary_file = gr.Button("Download reviewed summary", interactive=False)
            review_summary_file.click(fn=None, inputs=[review_summary_payload], outputs=[review_download_status],
                                      js=csv_download_js("ir_reviewed_summary.csv"), queue=False)
            review_score_button.click(review_scores, [], [review_table, review_results_payload, review_summary_payload, review_results_file, review_summary_file])

try:
    app.launch(share=False, theme=theme, css=css, show_error=True, allowed_paths=[str(REVIEW_EXPORTS)] if SHOW_REVIEW_TOOLS else [])
except TypeError:
    app.launch(share=False, show_error=True, allowed_paths=[str(REVIEW_EXPORTS)] if SHOW_REVIEW_TOOLS else [])
